# GALR genes vs immune cell infiltration and immune checkpoint expression

**Part A**: correlates GALR1/2/3 expression with immune cell infiltration
fractions (TIMER, falling back to CIBERSORT/xCell/MCP-counter
if TIMER columns aren't present) across TCGA cancer types.

**Part B**: correlates GALR1/2/3 expression with a panel of immune
checkpoint genes, pan-cancer and per cancer type.

All correlations use Spearman's r with BH-FDR correction.

**Inputs** (see `DATA_DIR` below):
- `tcga_RSEM_gene_tpm_annotated.tsv` — TCGA pan-cancer TPM expression matrix
- `infiltration_estimation_for_tcga.csv.gz` — immune infiltration estimates (TIMER2.0 export)
- `TCGA_sample_metadata.tsv` — sample -> cancer type mapping
- `selected_cancers.txt` — one cancer type per line, for the per-cancer plot

**Outputs** (written to `OUTPUT_DIR`):
- `GALR_immune_infiltration_pancancer.tsv`
- `GALR_checkpoint_pancancer.tsv`
- heatmaps (`.png`) for infiltration, checkpoint (pan-cancer), and checkpoint (per cancer)

In [ ]:
import os

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

## Configuration

Edit these for your own machine/dataset.

In [ ]:
DATA_DIR = "data"
OUTPUT_DIR = "output"

TARGET_GENES = ["GALR1", "GALR2", "GALR3"]
CHECKPOINT_GENES = ["CD274", "PDCD1", "CTLA4", "HAVCR2", "LAG3", "TIGIT", "PDCD1LG2", "CD276"]
FDR_CUTOFF = 0.05
MIN_SAMPLES_PANCANCER = 30
MIN_SAMPLES_PER_CANCER = 15

EXPRESSION_FILE = os.path.join(DATA_DIR, "tcga_RSEM_gene_tpm_annotated.tsv")
INFILTRATION_FILE = os.path.join(DATA_DIR, "infiltration_estimation_for_tcga.csv.gz")
SAMPLE_METADATA_FILE = os.path.join(DATA_DIR, "TCGA_sample_metadata.tsv")
SELECTED_CANCERS_FILE = os.path.join(DATA_DIR, "selected_cancers.txt")

TCGA_ABBREVIATIONS = {
    "breast invasive carcinoma": "BRCA",
    "lung adenocarcinoma": "LUAD",
    "lung squamous cell carcinoma": "LUSC",
    "colon adenocarcinoma": "COAD",
    "rectum adenocarcinoma": "READ",
    "prostate adenocarcinoma": "PRAD",
    "thyroid carcinoma": "THCA",
    "head & neck squamous cell carcinoma": "HNSC",
    "esophageal carcinoma": "ESCA",
    "stomach adenocarcinoma": "STAD",
    "liver hepatocellular carcinoma": "LIHC",
    "kidney renal clear cell carcinoma": "KIRC",
    "kidney renal papillary cell carcinoma": "KIRP",
    "kidney chromophobe": "KICH",
    "bladder urothelial carcinoma": "BLCA",
    "uterine corpus endometrial carcinoma": "UCEC",
    "ovarian serous cystadenocarcinoma": "OV",
    "skin cutaneous melanoma": "SKCM",
    "pancreatic adenocarcinoma": "PAAD",
    "glioblastoma multiforme": "GBM",
    "brain lower grade glioma": "LGG",
}

## Load expression data

In [ ]:
def load_expression(path: str) -> pd.DataFrame:
    expr_df = pd.read_csv(path, sep="\t", index_col=0)
    if "hgnc_symbol" in expr_df.columns:
        expr_df = expr_df.set_index("hgnc_symbol")
    expr_df = expr_df.drop(columns=["Ensembl_clean", "sample"], errors="ignore")
    expr_df = expr_df.select_dtypes(include=["number"])
    expr_df = expr_df[~expr_df.index.duplicated(keep="first")]
    expr_df.columns = expr_df.columns.str[:15]
    print("Expression matrix:", expr_df.shape)
    return expr_df

## Load immune infiltration estimates

In [ ]:
def load_infiltration(path: str):
    """Load infiltration estimates, auto-detecting which method's columns are present."""
    inf_raw = pd.read_csv(path, index_col=0, compression="gzip")
    all_cols = inf_raw.columns.tolist()

    timer_cols = [c for c in all_cols if c.endswith("_TIMER")]
    cibersort_cols = [c for c in all_cols if "CIBERSORT" in c.upper() and "ABS" not in c.upper()]
    xcell_cols = [c for c in all_cols if "XCELL" in c.upper()]

    if timer_cols:
        immune_cols, method_label = timer_cols, "TIMER"
    elif cibersort_cols:
        immune_cols, method_label = cibersort_cols, "CIBERSORT"
    elif xcell_cols:
        immune_cols, method_label = xcell_cols, "xCELL"
    else:
        immune_cols = [
            c for c in all_cols
            if inf_raw[c].dtype in [float, np.float64] and inf_raw[c].between(0, 1).mean() > 0.8
        ]
        method_label = "unknown"
        print("Could not auto-detect infiltration method — using numeric 0-1 columns as proxy")

    print(f"Using {len(immune_cols)} immune columns ({method_label}): {immune_cols}")

    inf_df = inf_raw[immune_cols].copy()
    inf_df.index = inf_df.index.str[:15]
    inf_df = inf_df[~inf_df.index.duplicated(keep="first")]
    return inf_df, immune_cols, method_label

## Build merged table

In [ ]:
def build_merged_table(expr_df, target_genes, checkpoint_genes, inf_df, metadata_path):
    meta = pd.read_csv(metadata_path, sep="\t", dtype=str)
    meta = meta.set_index("sample").rename(columns={"cancer": "CancerType"})
    meta.index = meta.index.str[:15]

    genes_present = [g for g in target_genes if g in expr_df.index]
    ckpt_present = [g for g in checkpoint_genes if g in expr_df.index]
    print("Target genes found:", genes_present)
    print("Checkpoints found:", ckpt_present)

    expr_T = expr_df.loc[genes_present + ckpt_present].T
    merged = expr_T.join(inf_df, how="inner")
    merged["CancerType"] = merged.index.map(meta["CancerType"])
    merged = merged.dropna(subset=["CancerType"])
    print(f"Merged samples: {merged.shape[0]} | Cancer types: {merged['CancerType'].nunique()}")

    return merged, genes_present, ckpt_present

## Generic correlation helper

In [ ]:
def correlate(merged, genes, columns, min_samples, fdr_cutoff, gene_col_name, other_col_name):
    """Generic Spearman-correlate `genes` against `columns`, with global BH-FDR."""
    records = []
    for gene in genes:
        for col in columns:
            valid = merged[[gene, col]].dropna()
            if other_col_name == "ImmuneCell":
                valid = valid[valid[col] >= 0]  # drop negative CIBERSORT artefacts
            if len(valid) >= min_samples:
                r, p = spearmanr(valid[gene], valid[col])
                records.append({gene_col_name: gene, other_col_name: col,
                                 "Spearman_r": r, "p_value": p, "n": len(valid)})

    result = pd.DataFrame(records)
    if len(result) > 1:
        _, fdr, _, _ = multipletests(result["p_value"], method="fdr_bh")
        result["FDR"] = fdr
        result["Significant"] = result["FDR"] < fdr_cutoff
    return result

## Plotting functions

In [ ]:
def plot_infiltration_heatmap(inf_corr, genes_present, immune_cols, method_label, out_dir):
    label_map = {c: c.replace(f"_{method_label}", "").replace("_", " ") for c in immune_cols}

    for gene in genes_present:
        sub = inf_corr[inf_corr["Gene"] == gene].copy()
        sub["Label"] = sub["ImmuneCell"].map(label_map)
        sub = sub.set_index("Label")

        r_row = sub["Spearman_r"].reindex([label_map[c] for c in immune_cols])
        sig_row = sub["Significant"].reindex([label_map[c] for c in immune_cols]).fillna(False)

        fig, ax = plt.subplots(figsize=(len(immune_cols) * 0.9 + 1.5, 2.5))
        sns.heatmap(r_row.to_frame().T, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                    annot=True, fmt=".2f", linewidths=0.5,
                    cbar_kws={"label": "Spearman r"}, ax=ax)

        for j, lbl in enumerate([label_map[c] for c in immune_cols]):
            if not sig_row.get(lbl, False):
                ax.text(j + 0.5, 0.5, "x", ha="center", va="center", color="black", fontsize=12)

        ax.set_title(f"{gene} vs Immune Cell Infiltration — pan-cancer ({method_label})", fontsize=12)
        ax.set_yticklabels([gene], rotation=0)
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"{gene}_immune_infiltration_heatmap.png"),
                    dpi=300, bbox_inches="tight")
        plt.show()

In [ ]:
def plot_checkpoint_heatmap_pancancer(ckpt_corr, genes_present, ckpt_present, out_dir):
    pivot_r = ckpt_corr.pivot(index="Gene", columns="Checkpoint", values="Spearman_r")
    pivot_sig = ckpt_corr.pivot(index="Gene", columns="Checkpoint", values="Significant").fillna(False)

    fig, ax = plt.subplots(figsize=(len(ckpt_present) * 0.95 + 1, len(genes_present) * 1.0 + 1.5))
    sns.heatmap(pivot_r, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                annot=True, fmt=".2f", linewidths=0.5,
                cbar_kws={"label": "Spearman r"}, ax=ax)

    for i, gene in enumerate(pivot_r.index):
        for j, ckpt in enumerate(pivot_r.columns):
            if not pivot_sig.loc[gene, ckpt]:
                ax.text(j + 0.5, i + 0.5, "x", ha="center", va="center", color="black", fontsize=11)

    ax.set_title("Target Genes vs Immune Checkpoints — pan-cancer", fontsize=13)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.figtext(0.5, -0.03, "x = FDR >= 0.05", ha="center", fontsize=10)
    plt.savefig(os.path.join(out_dir, "GALR_checkpoint_heatmap.png"), dpi=300, bbox_inches="tight")
    plt.show()
    return pivot_r

In [ ]:
def plot_checkpoint_per_cancer(merged, ckpt_corr, ckpt_present, selected_cancers,
                                min_samples, fdr_cutoff, out_dir):
    """Per-cancer checkpoint correlation heatmap for whichever gene has the most significant hits."""
    best_gene = (
        ckpt_corr[ckpt_corr["Significant"]]["Gene"].value_counts().idxmax()
        if "Significant" in ckpt_corr and ckpt_corr["Significant"].any()
        else ckpt_corr["Gene"].iloc[0]
    )

    per_r = pd.DataFrame(index=ckpt_present, columns=selected_cancers, dtype=float)
    per_sig = pd.DataFrame(False, index=ckpt_present, columns=selected_cancers)
    all_p, all_idx = [], []

    for cancer in selected_cancers:
        sub = merged[merged["CancerType"] == cancer]
        for ckpt in ckpt_present:
            valid = sub[[best_gene, ckpt]].dropna()
            if len(valid) >= min_samples:
                r, p = spearmanr(valid[best_gene], valid[ckpt])
                per_r.loc[ckpt, cancer] = r
                all_p.append(p)
                all_idx.append((ckpt, cancer))

    if all_p:
        _, fdr_v, _, _ = multipletests(all_p, method="fdr_bh")
        for (ckpt, cancer), fv in zip(all_idx, fdr_v):
            if fv < fdr_cutoff:
                per_sig.loc[ckpt, cancer] = True

    tcga_labels = [TCGA_ABBREVIATIONS.get(c.lower().strip(), c) for c in selected_cancers]
    per_r.columns = tcga_labels
    per_sig.columns = tcga_labels

    fig, ax = plt.subplots(figsize=(max(12, len(tcga_labels) * 0.85), len(ckpt_present) * 0.7 + 2))
    sns.heatmap(per_r.astype(float), cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                annot=True, fmt=".2f", linewidths=0.5,
                cbar_kws={"label": "Spearman r"}, ax=ax)

    for i, ckpt in enumerate(ckpt_present):
        for j, cancer in enumerate(tcga_labels):
            if not per_sig.iloc[i, j]:
                ax.text(j + 0.5, i + 0.5, "x", ha="center", va="center", color="black", fontsize=9)

    ax.set_title(f"{best_gene} vs Immune Checkpoints per Cancer Type", fontsize=13)
    plt.xticks(rotation=60, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.figtext(0.5, -0.03, "x = FDR >= 0.05", ha="center", fontsize=10)
    plt.savefig(os.path.join(out_dir, f"{best_gene}_checkpoint_per_cancer.png"),
                dpi=300, bbox_inches="tight")
    plt.show()

## Run the analysis

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

expr_df = load_expression(EXPRESSION_FILE)
inf_df, immune_cols, method_label = load_infiltration(INFILTRATION_FILE)
merged, genes_present, ckpt_present = build_merged_table(
    expr_df, TARGET_GENES, CHECKPOINT_GENES, inf_df, SAMPLE_METADATA_FILE
)

with open(SELECTED_CANCERS_FILE) as f:
    selected_cancers = [c for c in f.read().strip().split("\n")
                         if c in merged["CancerType"].unique()]

## Part A: target genes vs immune cell infiltration

In [ ]:
inf_corr = correlate(merged, genes_present, immune_cols, MIN_SAMPLES_PANCANCER,
                      FDR_CUTOFF, gene_col_name="Gene", other_col_name="ImmuneCell")
inf_corr.to_csv(os.path.join(OUTPUT_DIR, "GALR_immune_infiltration_pancancer.tsv"),
                 sep="\t", index=False)
plot_infiltration_heatmap(inf_corr, genes_present, immune_cols, method_label, OUTPUT_DIR)

## Part B: target genes vs immune checkpoint expression

In [ ]:
ckpt_corr = correlate(merged, genes_present, ckpt_present, MIN_SAMPLES_PANCANCER,
                       FDR_CUTOFF, gene_col_name="Gene", other_col_name="Checkpoint")
ckpt_corr.to_csv(os.path.join(OUTPUT_DIR, "GALR_checkpoint_pancancer.tsv"),
                  sep="\t", index=False)
plot_checkpoint_heatmap_pancancer(ckpt_corr, genes_present, ckpt_present, OUTPUT_DIR)

In [ ]:
plot_checkpoint_per_cancer(merged, ckpt_corr, ckpt_present, selected_cancers,
                            MIN_SAMPLES_PER_CANCER, FDR_CUTOFF, OUTPUT_DIR)